# LFW Grad-CAM — 01. Origin embedding and leave-one-out templates

Pass A에서 선택된 **모든 이미지**의 raw 512D, raw norm, unit embedding을
저장한 뒤 같은 split·identity 안에서 자기 자신을 제외한 template을
만듭니다. singleton과 identity 누락 표본도 행은 유지하며 명시적으로
부적격 처리합니다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

EXECUTION = CONFIG["execution"]
MODEL_PROFILE = str(EXECUTION["model_profile"])
MODE = str(EXECUTION["mode"])
DATA_FRACTION = float(EXECUTION["data_fraction"])
SEED = int(EXECUTION["seed"])
EXECUTE_STAGE = bool(EXECUTION["execute_stage"])
WRITE_OUTPUTS = bool(EXECUTION["write_outputs"])
OVERWRITE = bool(EXECUTION["overwrite"])

if MODEL_PROFILE not in CONFIG["models"]["selected_profiles"]:
    raise ValueError(f"선택되지 않은 모델 profile: {MODEL_PROFILE}")
MODEL_NAME = str(CONFIG["models"]["profiles"][MODEL_PROFILE]["family"])
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [ ]:
from research.runtime import RunStore, resolve_active_dataset_run

STEP2_RUN_DIR = resolve_active_dataset_run(
    PROJECT_ROOT / CONFIG["run"]["root"],
    dataset_id=str(CONFIG["run"]["dataset_id"]),
    directory_template=str(CONFIG["run"]["dataset_date_dir_template"]),
)
RUN = RunStore.open(STEP2_RUN_DIR)
WORKFLOW_ROOT = (
    STEP2_RUN_DIR / CONFIG["workflow"]["artifact_subdir"]
)

import json

import numpy as np
import pandas as pd

from research.embeddings import (
    create_pytorch_adapter_from_spec,
    select_model_spec,
)
from research.explainability.gradcam import (
    prepare_population_saliency_inputs,
    write_prepared_population_artifact,
)

FREEZE_MANIFEST_PATH = WORKFLOW_ROOT / CONFIG["workflow"]["freeze_manifest_path"]
SELECTED_MANIFEST_PATH = WORKFLOW_ROOT / CONFIG["workflow"]["selected_manifest_path"]
ALIGNED_FACES_NPY_PATH = PROJECT_ROOT / CONFIG["aligned_crops"]["faces_path"]
MODEL_REGISTRY_ROOT = PROJECT_ROOT / CONFIG["models"]["registry_root"]
PREPARED_ARTIFACT_OUTPUT_DIR = WORKFLOW_ROOT / CONFIG["workflow"]["prepared_population_dir"]
DEVICE = str(EXECUTION["device"])


In [ ]:
if EXECUTE_STAGE:
    paths = {
        "freeze": FREEZE_MANIFEST_PATH,
        "selected": SELECTED_MANIFEST_PATH,
        "aligned_faces": ALIGNED_FACES_NPY_PATH,
    }
    missing = [name for name, value in paths.items() if value is None]
    if missing:
        raise RuntimeError(f"입력 경로가 비어 있습니다: {missing}")
    freeze = json.loads(
        Path(paths["freeze"]).read_text(encoding="utf-8")
    )
    selected = pd.read_csv(paths["selected"])
    model_spec_path, spec = select_model_spec(
        MODEL_REGISTRY_ROOT,
        family=MODEL_NAME,
        model_uid=str(freeze["model_uid"]),
        verify_checkpoint=True,
    )
    if len(selected) != int(freeze["selected_sample_count"]):
        raise ValueError("동결된 선택 표본 수와 manifest가 다릅니다.")

    source_faces = np.load(
        paths["aligned_faces"],
        mmap_mode="r",
        allow_pickle=False,
    )
    indices = selected["aligned_face_index"].to_numpy(dtype=np.int64)
    aligned_faces = np.asarray(source_faces[indices], dtype=np.uint8)
    adapter = create_pytorch_adapter_from_spec(spec, device=DEVICE)
    prepared = prepare_population_saliency_inputs(
        adapter,
        aligned_faces,
        sample_ids=selected["sample_id"].astype(str),
        identity_ids=selected["identity_id"],
        scope_ids=selected["template_scope_id"].astype(str),
        extraction_uid=freeze["extraction_uid"],
        dataset_id="lfw",
        embedding_batch_size=int(
            CONFIG["gradcam"]["extraction"]["embedding_batch_size"]
        ),
        require_all_eligible=False,
    )
    coverage = prepared.loo_templates.coverage_summary()
    if len(prepared.sample_ids) != len(selected):
        raise RuntimeError("Pass A가 일부 선택 표본을 누락했습니다.")
    if WRITE_OUTPUTS:
        if PREPARED_ARTIFACT_OUTPUT_DIR is None:
            raise RuntimeError("PREPARED_ARTIFACT_OUTPUT_DIR를 지정하세요.")
        write_prepared_population_artifact(
            prepared,
            PREPARED_ARTIFACT_OUTPUT_DIR,
            shard_size=int(
                CONFIG["gradcam"]["extraction"]["shard_size"]
            ),
            overwrite=OVERWRITE,
        )
else:
    coverage = pd.DataFrame(
        [{"status": "not_executed", "reason": "EXECUTE_STAGE=False"}]
    )
coverage


LFW의 singleton은 오류가 아니라 target 정의의 한계입니다. 임베딩은
유지하되 다른 target으로 바꾸지 않으며, 이후 집단 통계에서는 eligibility를
반드시 함께 보고합니다.
